# Session Dashboard (Observability MCP)

Visualize research session metrics, traces, and trends.

## Learning Objectives
- Query session metrics from PostgreSQL
- Visualize iteration quality progression
- Analyze cost and token usage patterns
- Monitor failure frequency trends
- Monitor harness performance across sessions

In [ ]:
import os
import sys
import httpx
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from dotenv import load_dotenv
env_path = os.path.join('..', '.env')
load_dotenv(env_path, override=True)

_VERIFY_SSL = os.getenv("VERIFY_SSL", "true").lower() not in ("false", "0", "no")
_http_client = None if _VERIFY_SSL else httpx.Client(verify=False, timeout=httpx.Timeout(300.0))

from harness.metrics import MetricsAggregator
from harness.session import SessionManager
print("Dashboard modules loaded")

## 1. Session Overview

In [ ]:
# List recent research sessions
manager = SessionManager()
sessions = manager.list_sessions(limit=10)

if sessions:
    print(f"Recent Research Sessions ({len(sessions)} found):")
    print(f"{'ID':<14} {'Query':<40} {'Status':<10} {'Score':<6} {'Created'}")
    print("-" * 90)
    for s in sessions:
        print(f"{s['session_id']:<14} {s['query'][:38]:<40} {s['status']:<10} {s['quality_score']:<6.1f} {s['created_at'][:16]}")
else:
    print("No sessions found. Run the orchestrator first.")

## 2. Iteration Quality Progression

In [ ]:
# Simulate a metrics aggregator with sample data
metrics = MetricsAggregator(session_id="demo")

# Simulate 3 iterations
iterations_data = [
    (1, 2500, 800, 4.5, False),
    (2, 2800, 1000, 6.8, False),
    (3, 2200, 700, 8.2, True),
]

for it_num, inp_tokens, out_tokens, quality, passed in iterations_data:
    metrics.start_iteration(it_num)
    metrics.record_llm_call(inp_tokens, out_tokens, "granite-3.3-8b-instruct")
    metrics.record_llm_call(inp_tokens // 2, out_tokens // 2, "granite-3.3-8b-instruct")
    metrics.record_mcp_call(latency_ms=150)
    metrics.end_iteration(quality, passed)

summary = metrics.summary()
print("Session Metrics Summary:")
print(f"  Total iterations: {summary['iterations']}")
print(f"  Total tokens: {summary['total_tokens']}")
print(f"  Total cost: ${summary['total_cost_usd']:.4f}")
print(f"  Final score: {summary['final_score']}/10")
print(f"  Passed: {summary['passed']}")
print(f"\nPer-iteration breakdown:")
print(f"{'Iter':<6} {'Tokens':<8} {'Cost':<10} {'LLM Calls':<11} {'Quality':<9} {'Passed'}")
print("-" * 55)
for it in summary['per_iteration']:
    print(f"{it['iteration']:<6} {it['tokens']:<8} ${it['cost_usd']:<9.4f} {it['llm_calls']:<11} {it['quality']:<9.1f} {it['passed']}")

## 3. Token Usage Analysis

In [ ]:
# Visualize token usage across iterations (text-based chart)
print("\nToken Usage by Iteration:")
print("=" * 50)
max_tokens = max(it['tokens'] for it in summary['per_iteration'])
for it in summary['per_iteration']:
    bar_len = int(it['tokens'] / max_tokens * 40)
    bar = '█' * bar_len
    print(f"  Iter {it['iteration']}: {bar} {it['tokens']}")

print(f"\nQuality Progression:")
print("=" * 50)
for it in summary['per_iteration']:
    bar_len = int(it['quality'] / 10 * 40)
    bar = '▓' * bar_len + '░' * (40 - bar_len)
    status = '✓' if it['passed'] else '✗'
    print(f"  Iter {it['iteration']}: {bar} {it['quality']:.1f}/10 {status}")

## 4. Failure Trend Analysis

In [ ]:
# Query failure patterns from PostgreSQL
import psycopg2

try:
    conn = psycopg2.connect(
        host=os.getenv("PGVECTOR_HOST", "localhost"),
        port=os.getenv("PGVECTOR_PORT", "5432"),
        dbname=os.getenv("PGVECTOR_DB", "doc_research"),
        user=os.getenv("PGVECTOR_USER", "postgres"),
        password=os.getenv("PGVECTOR_PASSWORD", "postgres"),
    )
    cur = conn.cursor()
    cur.execute("""
        SELECT category, COUNT(*) as cnt
        FROM failure_log
        GROUP BY category
        ORDER BY cnt DESC
        LIMIT 10
    """)
    rows = cur.fetchall()
    cur.close()
    conn.close()
    
    if rows:
        print("Failure Frequency (All Sessions):")
        print("=" * 50)
        max_count = max(r[1] for r in rows)
        for category, count in rows:
            bar_len = int(count / max_count * 30)
            bar = '█' * bar_len
            print(f"  {category:<25} {bar} ({count})")
    else:
        print("No failure data yet. Run some research sessions first.")
except Exception as e:
    print(f"Could not query failures: {e}")
    print("Make sure PostgreSQL is running and tables are created.")

## Summary

The dashboard provides:
- **Session overview**: Status, scores, and iteration counts
- **Quality progression**: How scores improve across iterations
- **Cost analysis**: Token usage and cost per iteration
- **Failure trends**: Common failure patterns for system improvement

In production, these metrics would feed into Grafana/Prometheus for real-time monitoring.